In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import zipfile
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 0. Baseline acc teat

In [ ]:
base_dir = '/content/drive/MyDrive/26sp_ML/PR1'
zip_path = '/content/drive/MyDrive/26sp_ML/PR1/GTSRB.zip'
extract_dir = '/content/GTSRB'
model_path = '/content/drive/MyDrive/26sp_ML/PR1/resnet18_gtsrb_best_full_model.pth'

os.makedirs(extract_dir, exist_ok=True)


# Unzip if needed
if not os.path.exists(os.path.join(extract_dir, 'Test.csv')):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)

# Read test CSV
test_csv_path = os.path.join(extract_dir, 'Test.csv')
test_df = pd.read_csv(test_csv_path)
print("Test shape:", test_df.shape)

In [ ]:
class GTSRBDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['Path'])
        label = int(row['ClassId'])

        image = Image.open(img_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, label

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_dataset = GTSRBDataset(test_df, extract_dir, transform=test_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load best acc model
model = torch.load(model_path, map_location=device, weights_only=False)
model = model.to(device)
model.eval()


# Test accuracy
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Correct / Total: {correct} / {total}")

# 1. FGSM dataset generation(eps=2,4,8,16/255)

In [ ]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from tqdm.auto import tqdm


class GTSRBFGSMTestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel_path = row['Path']
        img_path = os.path.join(self.root_dir, rel_path)
        label = int(row['ClassId'])

        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)

        return image, label, rel_path

fgsm_transform = transforms.Compose([
    transforms.Resize((224, 224)),           # FGSM perturbation을 픽셀 공간 [0,1]에서 하려고 일부러 정규화 안 함
    transforms.ToTensor()
])

fgsm_dataset = GTSRBFGSMTestDataset(test_df, extract_dir, transform=fgsm_transform)
fgsm_loader = DataLoader(
    fgsm_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


## 1) FGSM 데이터셋 생성

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

eps_list_255 = [2, 4, 8, 16]
eps_list = [e / 255.0 for e in eps_list_255]
print("Test shape:", test_df.shape)
print("eps_list_255:", eps_list_255)


# Normalize function (모델 입력 직전 적용)
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
def normalize_batch(x):
    return (x - mean) / std

criterion = nn.CrossEntropyLoss()

# Save roots
fgsm_roots = {}
for e255 in eps_list_255:
    root = os.path.join(base_dir, f'GTSRB_FGSM_eps{e255}_255')
    os.makedirs(root, exist_ok=True)
    fgsm_roots[e255] = root
    print(f'FGSM root for eps={e255}/255 -> {root}')

In [ ]:
# FGSM generation

clean_correct = 0
total = 0

stats = {
    e255: {
        'adv_correct': 0,
        'attack_success_on_clean': 0
    }
    for e255 in eps_list_255
}

saved_paths = []
saved_labels = []

for images, labels, rel_paths in tqdm(fgsm_loader, desc='Generating FGSM (2,4,8,16)'):
    images = images.to(device, non_blocking=True)   # [0,1] pixel space
    labels = labels.to(device, non_blocking=True)

    images.requires_grad_(True)   # input gradient도 추적

    # clean forward
    model.zero_grad()
    outputs = model(normalize_batch(images))
    loss = criterion(outputs, labels)
    clean_preds = outputs.argmax(dim=1)        # 원본이미지 예측 클래스

    # input gradient
    loss.backward()
    grad_sign = images.grad.sign().detach()    # eps 전 sign

    clean_correct += (clean_preds == labels).sum().item()
    total += labels.size(0)

    # path / label 저장은 한 번만
    for i, rel_path in enumerate(rel_paths):
        saved_paths.append(rel_path)
        saved_labels.append(int(labels[i].item()))

    # eps별 adversarial image 생성
    for e255, eps in zip(eps_list_255, eps_list):
        adv_images = torch.clamp(images.detach() + eps * grad_sign, 0.0, 1.0)

        # adversarial image to model
        with torch.no_grad():
            adv_outputs = model(normalize_batch(adv_images))
            adv_preds = adv_outputs.argmax(dim=1)

        stats[e255]['adv_correct'] += (adv_preds == labels).sum().item()
        stats[e255]['attack_success_on_clean'] += ((clean_preds == labels) & (adv_preds != labels)).sum().item()

        root = fgsm_roots[e255]
        for i, rel_path in enumerate(rel_paths):
            save_path = os.path.join(root, rel_path)
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            save_image(adv_images[i].cpu(), save_path)

In [ ]:
# Save Test.csv / config / summary

fgsm_test_df = pd.DataFrame({'Path': saved_paths, 'ClassId': saved_labels})

summary_rows = []
clean_acc = clean_correct / total

for e255 in eps_list_255:
    root = fgsm_roots[e255]

    csv_path = os.path.join(root, 'Test.csv')
    fgsm_test_df.to_csv(csv_path, index=False)

    config_path = os.path.join(root, 'attack_config.txt')
    with open(config_path, 'w') as f:
        f.write('attack = FGSM\n')
        f.write(f'epsilon = {e255 / 255.0}\n')
        f.write(f'epsilon_255 = {e255}\n')
        f.write('image_space = pixel_space_[0,1]\n')
        f.write('model_input = normalized_with_imagenet_stats\n')

    adv_acc = stats[e255]['adv_correct'] / total
    asr = stats[e255]['attack_success_on_clean'] / clean_correct if clean_correct > 0 else 0.0

    summary_rows.append({
        'epsilon_255': e255,
        'clean_acc': clean_acc,
        'adv_acc': adv_acc,
        'attack_success_rate': asr,
        'save_root': root
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv_path = os.path.join(base_dir, 'fgsm_multi_eps_summary.csv')
summary_df.to_csv(summary_csv_path, index=False)

print("\n========== FGSM Generation Done ==========")
print(summary_df)
print(f"\nSummary saved to: {summary_csv_path}")
print("==========================================")

## 2) FGSM dataset 별 예측 정확도 측정

In [ ]:
# Eval Transform
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# Evaluation loop
results = []

for eps in eps_list_255:
    fgsm_root = os.path.join(base_dir, f'GTSRB_FGSM_eps{eps}_255')
    fgsm_csv_path = os.path.join(fgsm_root, 'Test.csv')

    if not os.path.exists(fgsm_csv_path):
        print(f"[SKIP] eps={eps}/255 -> Test.csv not found: {fgsm_csv_path}")
        continue

    fgsm_df = pd.read_csv(fgsm_csv_path)

    fgsm_dataset = GTSRBDataset(fgsm_df, fgsm_root, transform=eval_transform)
    fgsm_loader = DataLoader(
        fgsm_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in fgsm_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total if total > 0 else 0.0

    print(f"eps={eps}/255 -> Accuracy: {acc:.4f} ({correct}/{total})")

    results.append({
        'epsilon_255': eps,
        'accuracy': acc,
        'correct': correct,
        'total': total,
        'dataset_root': fgsm_root
    })

# Summary
results_df = pd.DataFrame(results)

print("\n========== FGSM Evaluation Summary ==========")
print(results_df)
print("============================================")

summary_path = os.path.join(base_dir, 'fgsm_eval_summary_2_4_8_16.csv')
results_df.to_csv(summary_path, index=False)
print(f"\nSaved summary to: {summary_path}")

## 3) 각 데이터셋별 이미지 확인

In [ ]:
import matplotlib.pyplot as plt

roots = {
    'clean': extract_dir,
    'eps=2/255': os.path.join(base_dir, 'GTSRB_FGSM_eps2_255'),
    'eps=4/255': os.path.join(base_dir, 'GTSRB_FGSM_eps4_255'),
    'eps=8/255': os.path.join(base_dir, 'GTSRB_FGSM_eps8_255'),
    'eps=16/255': os.path.join(base_dir, 'GTSRB_FGSM_eps16_255'),
}

sample_idx = 0                      # 확인할 idx num
row = test_df.iloc[sample_idx]
rel_path = row['Path']
true_label = int(row['ClassId'])

def predict_image(pil_img):
    x = eval_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x).argmax(dim=1).item()
    return pred

fig, axes = plt.subplots(1, len(roots), figsize=(4.5 * len(roots), 5))

for ax, (name, root_dir) in zip(axes, roots.items()):
    img_path = os.path.join(root_dir, rel_path)

    if not os.path.exists(img_path):
        ax.set_title(f"{name}\nNOT FOUND")
        ax.axis('off')
        continue

    img = Image.open(img_path).convert('RGB')
    pred = predict_image(img)

    ax.imshow(img.resize((224, 224)))
    ax.set_title(f"{name}\nTrue: {true_label} | Pred: {pred}")
    ax.axis('off')

plt.tight_layout()
plt.show()

# 2. edge-fgsm dataset generation

In [ ]:
# edge-fgsm and random sampling settings
import random
import numpy as np
import torch.nn.functional as functional

edge_eps_list_255 = [8, 16, 24, 48]
fgsm_eps_list_255 = [2, 4, 8, 16]

edge_beta = 0.7
random_seed = 42
n_pixel_sample = 500
batch_size = 64
num_workers = 0

clean_root = extract_dir
clean_csv = os.path.join(clean_root, 'Test.csv')

if not os.path.exists(clean_csv):
    raise FileNotFoundError(f'Test.csv not found: {clean_csv}')

clean_df = test_df.reset_index(drop=True).copy()

print('clean_root:', clean_root)
print('clean_csv:', clean_csv)
print('clean test shape:', clean_df.shape)

## 1) edge map function

In [ ]:
def sobel_edge_map(x):
    gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]

    sobel_x = torch.tensor(
        [[[-1, 0, 1],
          [-2, 0, 2],
          [-1, 0, 1]]],
        dtype=x.dtype,
        device=x.device
    ).unsqueeze(0)

    sobel_y = torch.tensor(
        [[[-1, -2, -1],
          [ 0,  0,  0],
          [ 1,  2,  1]]],
        dtype=x.dtype,
        device=x.device
    ).unsqueeze(0)

    gx = functional.conv2d(gray, sobel_x, padding=1)
    gy = functional.conv2d(gray, sobel_y, padding=1)
    magnitude = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

    batch = magnitude.size(0)
    magnitude_flat = magnitude.view(batch, -1)
    magnitude_min = magnitude_flat.min(dim=1)[0].view(batch, 1, 1, 1)
    magnitude_max = magnitude_flat.max(dim=1)[0].view(batch, 1, 1, 1)

    edge_map = (magnitude - magnitude_min) / (magnitude_max - magnitude_min + 1e-8)
    return edge_map


def build_edge_weight_map(x, beta=edge_beta):
    edge_map = sobel_edge_map(x)
    weight_map = (1.0 - beta) + beta * edge_map
    weight_map = weight_map.repeat(1, 3, 1, 1)
    return edge_map, weight_map


def get_edge_root(eps_255):
    return os.path.join(base_dir, f'GTSRB_EDGE_FGSM_eps{eps_255}_255_beta0p7')


def get_fgsm_root(eps_255):
    return os.path.join(base_dir, f'GTSRB_FGSM_eps{eps_255}_255')


def save_edge_attack_config(root_dir, eps_255, beta=edge_beta):
    config_path = os.path.join(root_dir, 'attack_config.txt')

    with open(config_path, 'w') as file:
        file.write('attack = Edge-Weighted FGSM\n')
        file.write(f'epsilon = {eps_255 / 255.0}\n')
        file.write(f'epsilon_255 = {eps_255}\n')
        file.write(f'beta = {beta}\n')
        file.write('edge_detector = sobel_magnitude\n')
        file.write('weight_map = (1-beta) + beta * edge_map\n')
        file.write('image_space = pixel_space_[0,1]\n')
        file.write('model_input = normalized_with_imagenet_stats\n')

    return config_path

## 2) edge-FGSM generation

In [ ]:
edge_clean_dataset = GTSRBFGSMTestDataset(clean_df, clean_root, transform=fgsm_transform)
edge_clean_loader = DataLoader(
    edge_clean_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available()
)


def generate_edge_fgsm_dataset(eps_255, beta=edge_beta):
    epsilon = eps_255 / 255.0
    edge_root = get_edge_root(eps_255)
    os.makedirs(edge_root, exist_ok=True)

    saved_paths = []
    saved_labels = []

    clean_correct = 0
    attack_correct = 0
    attack_success_on_clean = 0
    total = 0

    model.eval()

    for images, labels, rel_paths in tqdm(edge_clean_loader, desc=f'Generating Edge-FGSM eps={eps_255}/255'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images.requires_grad_(True)
        model.zero_grad(set_to_none=True)

        clean_outputs = model(normalize_batch(images))
        loss = criterion(clean_outputs, labels)
        clean_preds = clean_outputs.argmax(dim=1)

        loss.backward()
        grad_sign = images.grad.detach().sign()

        with torch.no_grad():
            _, weight_map = build_edge_weight_map(images.detach(), beta=beta)
            attack_images = torch.clamp(
                images.detach() + epsilon * weight_map * grad_sign,
                min=0.0,
                max=1.0
            )

            attack_outputs = model(normalize_batch(attack_images))
            attack_preds = attack_outputs.argmax(dim=1)

        clean_correct += (clean_preds == labels).sum().item()
        attack_correct += (attack_preds == labels).sum().item()
        attack_success_on_clean += ((clean_preds == labels) & (attack_preds != labels)).sum().item()
        total += labels.size(0)

        for image_index, rel_path in enumerate(rel_paths):
            save_path = os.path.join(edge_root, rel_path)
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            save_image(attack_images[image_index].cpu(), save_path)

            saved_paths.append(rel_path)
            saved_labels.append(int(labels[image_index].item()))

    edge_df = pd.DataFrame({'Path': saved_paths, 'ClassId': saved_labels})
    edge_csv = os.path.join(edge_root, 'Test.csv')
    edge_df.to_csv(edge_csv, index=False)

    config_path = save_edge_attack_config(edge_root, eps_255, beta=beta)

    clean_acc = clean_correct / total if total > 0 else 0.0
    attack_acc = attack_correct / total if total > 0 else 0.0
    attack_success_rate = attack_success_on_clean / clean_correct if clean_correct > 0 else 0.0

    return {
        'attack': f'Edge-FGSM {eps_255}/255',
        'epsilon_255': eps_255,
        'beta': beta,
        'clean_acc_reference': clean_acc,
        'edge_fgsm_acc': attack_acc,
        'attack_success_rate_on_clean_correct': attack_success_rate,
        'total': total,
        'dataset_root': edge_root,
        'csv_path': edge_csv,
        'config_path': config_path
    }


edge_generation_rows = []

for eps_255 in edge_eps_list_255:
    edge_generation_rows.append(generate_edge_fgsm_dataset(eps_255, beta=edge_beta))

edge_generation_df = pd.DataFrame(edge_generation_rows)

print('\n========== edge-fgsm generation summary ==========')
print(edge_generation_df)
print('=================================================')

edge_generation_summary = os.path.join(base_dir, 'edge_fgsm_generation_summary_8_16_24_48.csv')
edge_generation_df.to_csv(edge_generation_summary, index=False)
print('saved summary to:', edge_generation_summary)

# 3. accuracy evaluation

Edge-FGSM 8,16,24,48/255 과 FGSM 2,4,8,16/255의 정확도 측정

In [ ]:
def load_attack_df(root_dir):
    csv_path = os.path.join(root_dir, 'Test.csv')

    if os.path.exists(csv_path):
        return pd.read_csv(csv_path).reset_index(drop=True)

    return clean_df.copy()


def evaluate_attack_dataset(root_dir, attack_name):
    if not os.path.exists(root_dir):
        print(f'[skip] {attack_name}: folder not found -> {root_dir}')
        return {
            'attack': attack_name,
            'accuracy': np.nan,
            'correct': 0,
            'total': 0,
            'dataset_root': root_dir
        }

    attack_df = load_attack_df(root_dir)
    attack_dataset = GTSRBDataset(attack_df, root_dir, transform=eval_transform)
    attack_loader = DataLoader(
        attack_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    correct = 0
    total = 0
    model.eval()

    with torch.no_grad():
        for images, labels in tqdm(attack_loader, desc=f'Evaluating {attack_name}'):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total if total > 0 else np.nan
    print(f'{attack_name}: accuracy = {accuracy:.4f} ({correct}/{total})')

    return {'attack': attack_name, 'accuracy': accuracy, 'correct': correct, 'total': total, 'dataset_root': root_dir}


attack_roots = {}

for eps_255 in edge_eps_list_255:
    attack_roots[f'Edge-FGSM {eps_255}/255'] = get_edge_root(eps_255)

for eps_255 in fgsm_eps_list_255:
    attack_roots[f'FGSM {eps_255}/255'] = get_fgsm_root(eps_255)

accuracy_rows = []

for attack_name, root_dir in attack_roots.items():
    accuracy_rows.append(evaluate_attack_dataset(root_dir, attack_name))

attack_accuracy_df = pd.DataFrame(accuracy_rows)

print('\n========== attack accuracy summary ==========')
print(attack_accuracy_df)
print('============================================')

attack_accuracy_summary = os.path.join(base_dir, 'edge_fgsm_and_fgsm_accuracy_summary.csv')
attack_accuracy_df.to_csv(attack_accuracy_summary, index=False)
print('saved summary to:', attack_accuracy_summary)

# 4. mean pixel-change evaluation

동알헌 랜덤 500개 인덱스를 기준으로 각 공격 이미지의 평균 픽셀 변화량 측정

In [ ]:
random.seed(random_seed)

num_samples = min(n_pixel_sample, len(clean_df))
pixel_sample_indices = random.sample(range(len(clean_df)), num_samples)
pixel_sample_df = clean_df.iloc[pixel_sample_indices].reset_index(drop=True)

pixel_sample_csv = os.path.join(base_dir, f'pixel_change_sample_indices_{num_samples}_seed{random_seed}.csv')
pixel_sample_df.assign(original_index=pixel_sample_indices).to_csv(pixel_sample_csv, index=False)
print('saved sampled indices to:', pixel_sample_csv)


def load_pixel_tensor(root_dir, rel_path):
    image_path = os.path.join(root_dir, rel_path)
    image = Image.open(image_path).convert('RGB')
    return fgsm_transform(image)


def compute_pixel_change_for_attack(attack_name, attack_root, subset_df, original_indices):
    if not os.path.exists(attack_root):
        print(f'[skip] {attack_name}: folder not found -> {attack_root}')
        return pd.DataFrame()

    rows = []

    for local_index, row in tqdm(
        enumerate(subset_df.itertuples(index=False)),
        total=len(subset_df),
        desc=f'Pixel change {attack_name}'
    ):
        rel_path = row.Path
        label = int(row.ClassId)

        clean_path = os.path.join(clean_root, rel_path)
        attack_path = os.path.join(attack_root, rel_path)

        if not os.path.exists(clean_path) or not os.path.exists(attack_path):
            continue

        clean_x = load_pixel_tensor(clean_root, rel_path)
        attack_x = load_pixel_tensor(attack_root, rel_path)
        diff = (attack_x - clean_x).abs()

        rows.append({
            'attack': attack_name,
            'original_index': original_indices[local_index],
            'Path': rel_path,
            'ClassId': label,
            'mean_abs_pixel_change_per_image': diff.mean().item(),
            'max_abs_pixel_change_per_image': diff.max().item(),
            'rmse_per_image': torch.sqrt((diff ** 2).mean()).item()
        })

    return pd.DataFrame(rows)


pixel_detail_dfs = []
pixel_summary_rows = []

for attack_name, attack_root in attack_roots.items():
    detail_df = compute_pixel_change_for_attack(
        attack_name,
        attack_root,
        pixel_sample_df,
        pixel_sample_indices
    )

    if len(detail_df) == 0:
        pixel_summary_rows.append({
            'attack': attack_name,
            'num_samples_used': 0,
            'mean_of_mean_abs_pixel_change': np.nan,
            'std_of_mean_abs_pixel_change': np.nan,
            'mean_of_max_abs_pixel_change': np.nan,
            'mean_rmse': np.nan,
            'dataset_root': attack_root
        })
        continue

    pixel_detail_dfs.append(detail_df)
    pixel_summary_rows.append({
        'attack': attack_name,
        'num_samples_used': len(detail_df),
        'mean_of_mean_abs_pixel_change': detail_df['mean_abs_pixel_change_per_image'].mean(),
        'std_of_mean_abs_pixel_change': detail_df['mean_abs_pixel_change_per_image'].std(),
        'mean_of_max_abs_pixel_change': detail_df['max_abs_pixel_change_per_image'].mean(),
        'mean_rmse': detail_df['rmse_per_image'].mean(),
        'dataset_root': attack_root
    })

pixel_change_summary_df = pd.DataFrame(pixel_summary_rows)

print('\n========== pixel change summary ==========')
print(pixel_change_summary_df)
print('=========================================')

pixel_change_summary = os.path.join(base_dir, f'pixel_change_summary_{num_samples}_seed{random_seed}.csv')
pixel_change_summary_df.to_csv(pixel_change_summary, index=False)
print('saved summary to:', pixel_change_summary)

if len(pixel_detail_dfs) > 0:
    pixel_change_detail_df = pd.concat(pixel_detail_dfs, ignore_index=True)
    pixel_change_detail = os.path.join(base_dir, f'pixel_change_detail_{num_samples}_seed{random_seed}.csv')
    pixel_change_detail_df.to_csv(pixel_change_detail, index=False)
    print('saved detail to:', pixel_change_detail)

# 5. same-index image comparison

Original + Edge-FGSM 8,16,24,48/255 + FGSM 2,4,8,16/255 이미지를 같은 인덱스로 출력

In [ ]:
import math

base_dir = '/content/drive/MyDrive/26sp_ML/PR1'

clean_root = '/content/GTSRB'
clean_csv = os.path.join(clean_root, 'Test.csv')

image_roots = {
    'Original': clean_root,
    'FGSM 2/255': os.path.join(base_dir, 'GTSRB_FGSM_eps2_255'),
    'FGSM 4/255': os.path.join(base_dir, 'GTSRB_FGSM_eps4_255'),
    'FGSM 8/255': os.path.join(base_dir, 'GTSRB_FGSM_eps8_255'),
    'FGSM 16/255': os.path.join(base_dir, 'GTSRB_FGSM_eps16_255'),
    'Edge-FGSM 8/255': os.path.join(base_dir, 'GTSRB_EDGE_FGSM_eps8_255_beta0p7'),
    'Edge-FGSM 16/255': os.path.join(base_dir, 'GTSRB_EDGE_FGSM_eps16_255_beta0p7'),
    'Edge-FGSM 24/255': os.path.join(base_dir, 'GTSRB_EDGE_FGSM_eps24_255_beta0p7'),
    'Edge-FGSM 48/255': os.path.join(base_dir, 'GTSRB_EDGE_FGSM_eps48_255_beta0p7'),
}

test_df = pd.read_csv(clean_csv)

sample_idx = 0  # 여기만 바꿔가면서 확인

row = test_df.iloc[sample_idx]
rel_path = row['Path']
true_label = int(row['ClassId'])

print(f'sample_idx : {sample_idx}')
print(f'rel_path   : {rel_path}')
print(f'true_label : {true_label}')

items = list(image_roots.items())
n_cols = 3
n_rows = math.ceil(len(items) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4.2 * n_rows))
axes = axes.flatten()

for ax, (name, root_dir) in zip(axes, items):
    image_path = os.path.join(root_dir, rel_path)

    if not os.path.exists(image_path):
        ax.set_title(f'{name}\nNOT FOUND')
        ax.axis('off')
        continue

    image = Image.open(image_path).convert('RGB')
    image = image.resize((224, 224))

    ax.imshow(image)
    ax.set_title(f'{name}\nTrue: {true_label}')
    ax.axis('off')

for ax in axes[len(items):]:
    ax.axis('off')

plt.suptitle(f'Same-index comparison | index={sample_idx} | path={rel_path}', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 9))

fgsm_df = summary_df[summary_df['method'] == 'FGSM'].sort_values('mean_abs_pixel_change')
edge_df = summary_df[summary_df['method'] == 'Edge-FGSM'].sort_values('mean_abs_pixel_change')

plt.plot(fgsm_df['mean_abs_pixel_change'], fgsm_df['accuracy'], marker='o', label='FGSM')
plt.plot(edge_df['mean_abs_pixel_change'], edge_df['accuracy'], marker='o', label='Edge-FGSM')

for _, row in summary_df.iterrows():
    plt.annotate(
        row['attack'],
        (row['mean_abs_pixel_change'], row['accuracy']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

plt.xlabel('Mean perturbation per image')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Mean Perturbation')
plt.grid(True)
plt.legend()
plt.show()